# 8 — Uncertainty and the limits of the model

Companion to **section 9**. A capacity expansion is one answer to one weather
year at one set of costs. The section asks three questions of it: how much of
the answer is the *weather*, how much is the *cost projection*, and what
happens to a tax and a cap when the weather is not what they were set for.

Every experiment re-solves section 7's Danish greenfield, so this notebook is
the slow one: seven solves. At 96 segments each takes under
a minute; the note's own sweeps run at 1,095 through `run_weather.py`,
`run_costs.py` and `run_taxcap.py`.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
sys.path.insert(0, str(note / "pipeline"))   # the run scripts' helpers
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

In [ ]:
# The investment models read their cost tables when imported, so a missing
# table shows up here rather than at the first solve.
try:
    from model import greenfield
except FileNotFoundError as missing:
    raise SystemExit(
        f"Missing {Path(missing.filename).name}. This file is not shipped with the "
        "repository: it is our reshaped subset of an external technology-cost "
        "database whose compiled outputs carry no single stated licence, so you "
        "build it yourself, once, with\n\n    python data/prepare.py --costs-only\n\n"
        "from the note's directory (one small download per vintage, at a pinned "
        "version). Notebooks 00-05 and 09 run without it."
    ) from None

costs = pd.read_csv(PROCESSED / f"technology_costs_full_{greenfield.FORWARD_HORIZON}.csv",
                    index_col="technology")
print(f"cost table: {len(costs)} technologies, {greenfield.FORWARD_HORIZON} vintage")

## The weather

The zonal archive has ten consistent ERA5 years, 2015–2024, for all twelve
zones at once — consistent because a becalmed Denmark paired with an
unrelated German year would import its way out of every scarcity hour. Solve
the same system on three of them.

In [ ]:
from run_greenfield import year_weather, REFERENCE_WEATHER_YEAR

HOURS = 96
NC = PROCESSED / "network_eur_bz_2024.nc"
YEARS = [2016, 2021, REFERENCE_WEATHER_YEAR]

def dk_bill(n):
    """The Danish generators' share of the objective: their capacity charges
    plus their operating cost, EUR per year (batteries and trade left out --
    run_greenfield.py's mix_row has the full account)."""
    w = n.snapshot_weightings.objective
    g = n.generators[n.generators.bus.isin(greenfield.DK_ZONES)]
    fixed = (g["p_nom_opt"] * g["capital_cost"]).sum()
    mc = n.get_switchable_as_dense("Generator", "marginal_cost")[g.index]
    var = (n.generators_t.p[g.index] * mc).mul(w, axis=0).sum().sum()
    return fixed + var

solved = {}
for year in YEARS:
    m = greenfield.build(NC, costs, hours=HOURS, weather=year_weather(year))
    greenfield.solve(m)
    solved[year] = m
    print(f"{year}: Danish bill {dk_bill(m)/1e9:.2f} bnEUR, "
          f"emissions {greenfield.dk_emissions(m)/1e6:.2f} Mt")

mix = pd.DataFrame({y: greenfield.dk_capacity(m) for y, m in solved.items()}).fillna(0)
mix.round(0)

Same costs, same demand, same network. The spread across columns is the
section's first point: a single-year answer carries spurious precision, and
the *mix* moves more than the *bill* does — the system has several ways of
being cheap.

## The costs

`data/prepare.py --costs-only` also writes a pessimistic and an optimistic
reading of the 2050 cost projection (half the 2025→2050 trend added or
subtracted, per technology). Re-solve the reference year on the pessimistic
one.

In [ ]:
pess_file = PROCESSED / f"technology_costs_full_{greenfield.FORWARD_HORIZON}_pessimistic.csv"
if pess_file.exists():
    pess = pd.read_csv(pess_file, index_col="technology")
    m = greenfield.build(NC, costs_full=pess, hours=HOURS, weather=year_weather(REFERENCE_WEATHER_YEAR))
    greenfield.solve(m)
    ref = solved[REFERENCE_WEATHER_YEAR]
    print(f"Danish bill: baseline {dk_bill(ref)/1e9:.2f} bnEUR, pessimistic {dk_bill(m)/1e9:.2f} bnEUR")
    cost_table = pd.DataFrame({"baseline": greenfield.dk_capacity(ref),
                               "pessimistic costs": greenfield.dk_capacity(m)}).fillna(0).round(0)
else:
    cost_table = None
    print(f"{pess_file.name} not found -- run `python data/prepare.py --costs-only` to build the scenarios")
cost_table

## Tax versus cap

Section 2 proved a tax and a cap equivalent — for a *known* system. Set a
Danish budget in the reference year and read off its shadow price. Then move
to another weather year and impose (a) the same budget and (b) a tax equal to
that shadow price. Under the cap the *price* moves; under the tax the
*emissions* move. Which instrument you prefer depends on which of the two you
would rather not have move — Weitzman's argument, measured on this system.

In [ ]:
ref = solved[REFERENCE_WEATHER_YEAR]
budget = 0.10 * greenfield.dk_emissions(ref)

cap_ref = greenfield.build(NC, costs, hours=HOURS, weather=year_weather(REFERENCE_WEATHER_YEAR), co2_budget=budget)
tau = greenfield.solve(cap_ref)
print(f"reference year: budget {budget/1e6:.2f} Mt -> shadow price {tau:.0f} EUR/t")

other = YEARS[0]
cap = greenfield.build(NC, costs, hours=HOURS, weather=year_weather(other), co2_budget=budget)
sigma = greenfield.solve(cap)
tax = greenfield.build(NC, costs, hours=HOURS, weather=year_weather(other), carbon_tax=tau)
greenfield.solve(tax)

pd.DataFrame({
    "cap": [budget / 1e6, sigma],
    "tax": [greenfield.dk_emissions(tax) / 1e6, tau],
}, index=[f"emissions in {other} (Mt)", "carbon price (EUR/t)"]).round(2)

## The limits

Section 9 ends with what the model cannot see: demand that is also
weather-correlated but is not swapped here; a battery that knows the whole
year in advance (figure 9.6, from `run_storage.py`'s myopic experiment);
hydrogen and heat left out of the Danish zones; and a single scalar cost
projection standing in for a distribution. None of these is a bug. Each is a
place where the answer above is conditional on something the note states.

## Your turn

1. Run all ten weather years (each is one more solve) and plot the spread of
   the Danish bill against the spread of the optimal wind capacity. Which
   moves more, in percentage terms?
2. Repeat the tax-versus-cap comparison on the *system-wide* budget of
   notebook 7 (`expansion.build(..., carbon_tax=...)`). The note runs it
   there because that is where a cap binds hard.
3. The optimistic cost table is the other half of figure 9.4. Where does the
   Danish mix move when everything gets cheaper — and does the bill move
   less than the costs did?

In [ ]:
# Try it here.